# 05 — Hot-swap and disruption evidence

Internal swap phases, HTTP action duration, sink-observed output gap, event-aligned throughput dip, recovery, action duration, loss, and duplication are separate measurements. A smaller sink-observed gap does not imply a faster internal swap because queued output can mask internal disruption. E-Swap-4 contributes one event from each independent run. Explicit diagnostic inputs are thesis_evidence=false with descriptive-only uncertainty; missing inputs render PENDING with null values, never zero.


In [ ]:
import json, os, re
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.canonical import swap3_table, swap4_table
from wafer_analysis.focused import evidence_label, passed_artifacts, pending_record
from wafer_analysis.paths import resolve_analysis_batch
from wafer_analysis.plots import save_figure
from wafer_analysis.tables import save_table

def run_index(path):
    match=re.search(r'run-(\d+)',path.parent.name)
    if match is None: raise ValueError(f'malformed run name: {path.parent.name}')
    return int(match.group(1))

swap3_batch,swap3_canonical=resolve_analysis_batch('e-swap-3',os.environ.get('E_SWAP_3_DIR'))
swap3_runs=[]
if swap3_batch is not None:
    for path,value in passed_artifacts(swap3_batch,'disruption-analysis.json'): swap3_runs.append({**value,'run_index':run_index(path)})
if swap3_canonical: swap3=swap3_table(swap3_runs)
elif swap3_runs:
    raw3=pd.DataFrame(swap3_runs); swap3=(raw3.groupby('strategy').agg(N_runs=('run_index','nunique'),median_dip_percent=('dip_percent','median'),median_interruption_ns=('interruption_ns','median'),median_action_duration_ns=('action_duration_ns','median'),median_recovery_ns=('recovery_ns','median'),total_loss=('loss','sum'),total_duplicates=('duplicates','sum')).reset_index()); swap3['units']='percent, nanoseconds, messages'; swap3['estimator']='diagnostic run medians'; swap3['uncertainty']='descriptive only'; swap3['claim_boundary']='diagnostic stateless disruption'; swap3['thesis_evidence']=False
else: swap3=pd.DataFrame([pending_record('E-Swap-3 event-aligned disruption','no passed disruption-analysis.json leaf','percent, nanoseconds, messages')])
display(swap3)

swap4_batch,swap4_canonical=resolve_analysis_batch('e-swap-4',os.environ.get('E_SWAP_4_DIR'))
swap4_runs=[]
if swap4_batch is not None:
    for path,value in passed_artifacts(swap4_batch,'burst-timeline.json'): swap4_runs.append({**value,'run_index':run_index(path)})
if swap4_canonical: swap4=swap4_table(swap4_runs)
elif swap4_runs:
    gaps=[value['sink_observed_output_gap_ns'] for value in swap4_runs]; swap4=pd.DataFrame([{'experiment':'e-swap-4','condition':'burst-2x','N_runs':len(swap4_runs),'N_events':len(gaps),'median_sink_gap_ns':pd.Series(gaps).median(),'p95_sink_gap_ns':pd.Series(gaps).quantile(.95,interpolation='higher'),'total_loss':sum(value.get('loss',0) for value in swap4_runs),'total_duplicates':sum(value.get('sequence',{}).get('duplicates',0) for value in swap4_runs),'units':'nanoseconds, messages','estimator':'one sink gap per diagnostic run','uncertainty':'descriptive only','claim_boundary':'diagnostic one-burst evidence only','thesis_evidence':False}])
else: swap4=pd.DataFrame([pending_record('E-Swap-4 one-event burst','no passed burst-timeline.json leaf','nanoseconds, messages')])
display(swap4)

legacy=[]; seen=set()
for experiment in ('e-swap-1','e-swap-2','e-swap-6'):
    diagnostic=os.environ.get(f'{experiment.upper().replace("-","_")}_DIR')
    if experiment=='e-swap-1': diagnostic=diagnostic or os.environ.get('E_SWAP_DIR')
    batch,canonical=resolve_analysis_batch(experiment,diagnostic)
    if batch is None: continue
    for _,evidence in passed_artifacts(batch,'hotswap-analysis.json'):
        source=evidence.get('measurement_source_leaf')
        if source in seen: continue
        seen.add(source)
        for event in evidence.get('events',[]): legacy.append({'experiment':experiment,'condition':evidence.get('condition'),'source':source,'http_total_ms':event['http_total_ns']/1e6,'sink_gap_ms':event['sink_observed_output_gap_ns']/1e6})
if legacy:
    raw=pd.DataFrame(legacy); summary=(raw.groupby(['experiment','condition'],dropna=False).agg(N_runs=('source','nunique'),N_events=('source','size'),median_http_total_ms=('http_total_ms','median'),median_sink_gap_ms=('sink_gap_ms','median')).reset_index()); summary['units']='milliseconds'; summary['uncertainty']='descriptive only'; summary['thesis_evidence']=False; display(summary)
else: display(pd.DataFrame([pending_record('internal phases and sink-observed gap','no passed hotswap-analysis.json leaf','milliseconds')]))

if not swap3.empty and 'median_dip_percent' in swap3:
    fig,axes=plt.subplots(1,3,figsize=(15,4)); axes[0].bar(swap3.strategy,swap3.median_dip_percent); axes[0].axhline(5,linestyle='--',color='black'); axes[0].set_title('Event-aligned throughput dip'); axes[1].bar(swap3.strategy,swap3.median_action_duration_ns/1e6); axes[1].set_title('Control action duration'); axes[2].bar(swap3.strategy,swap3.median_recovery_ns/1e6); axes[2].set_title('Output recovery'); [ax.tick_params(axis='x',rotation=20) for ax in axes]
    output=os.environ.get('WAFER_ANALYSIS_OUTPUT_DIR')
    if output: save_figure(fig,'e-swap-3/disruption-metrics',output); save_table(swap3,'e-swap-3-disruption',output); save_table(swap4,'e-swap-4-burst',output)
print(f"{evidence_label(len(swap3_runs)+len(swap4_runs),'percent, nanoseconds, messages',swap3_canonical or swap4_canonical)}; internal phases, sink gaps, dip, action duration, recovery, and sequence integrity are not combined")
